In [ ]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

#Task 1

data = [
    ("How do I change my email?", "You can update it in your profile settings."),
    ("How can I reset my password?", "Use the password reset option on the login page."),
    ("My order is delayed", "Your order may take longer than expected."),
    ("Where can I track my order?", "You can track your order in your account."),
    ("How do I cancel my subscription?", "Go to billing settings and cancel your subscription."),
    ("Can I change my username?", "Yes, you can change it in profile settings."),
    ("How do I contact support?", "You can contact support through our help center."),
    ("I forgot my password", "Use password recovery on the login page."),
    ("Can I get a refund?", "Please submit a refund request through support."),
    ("How do I update my profile?", "Open account settings and edit your profile."),
    ("How do I delete my account?", "Contact support to permanently delete your account."),
    ("Payment failed", "Please verify your payment information and try again."),
    ("Where is my invoice?", "Invoices are available in billing settings."),
    ("How do I change language?", "You can select a new language in settings."),
    ("Can I download my data?", "Yes, export options are available in your account."),
    ("How do I enable notifications?", "Enable notifications from account settings."),
    ("Can I change my password?", "Yes, change it from security settings."),
    ("App is not working", "Please restart the application and try again."),
    ("How do I verify my account?", "Check your email and follow the verification link."),
    ("Can I update my payment method?", "Update payment information in billing settings."),
    ("How do I log out?", "Use the logout button in the menu."),
    ("Why is my account locked?", "Please contact support for assistance."),
    ("Can I recover deleted data?", "Recovery depends on available backups."),
    ("How do I create an account?", "Use the registration form on our website."),
    ("Can I change my phone number?", "Update your phone number in profile settings.")
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

questions = [preprocess(q) for q, a in data]
answers = [preprocess(a) for q, a in data]

all_text = " ".join(questions + answers)
tokens = all_text.split()

word_counts = Counter(tokens)

vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

for word in word_counts:
    vocab[word] = len(vocab)

idx_to_word = {v: k for k, v in vocab.items()}

max_len = 15

print("Vocabulary size:", len(vocab))


def encode(text):
    words = text.split()

    seq = [vocab.get(word, vocab["<UNK>"]) for word in words]

    seq = seq[:max_len]

    while len(seq) < max_len:
        seq.append(vocab["<PAD>"])

    return seq




class SupportDataset(Dataset):
    def __init__(self):
        self.samples = []

        for q, a in zip(questions, answers):

            input_seq = encode(q)
            target_seq = encode(a)

            self.samples.append(
                (
                    torch.tensor(input_seq),
                    torch.tensor(target_seq)
                )
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


dataset = SupportDataset()
loader = DataLoader(dataset, batch_size=4, shuffle=True)

#Task 2

class SupportFlowLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        output, _ = self.lstm(x)

        output = self.fc(output)

        return output


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SupportFlowLSTM(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 15

print("\nНавчання LSTM...\n")

for epoch in range(epochs):

    total_loss = 0

    for inputs, targets in loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs.view(-1, len(vocab)),
            targets.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} | Loss = {total_loss:.4f}"
    )


torch.save(
    model.state_dict(),
    "supportflow_lstm.pth"
)

print("\nLSTM модель збережена.")

#Task 3

def generate_response(question):

    question = preprocess(question)

    sequence = encode(question)

    x = torch.tensor([sequence]).to(device)

    with torch.no_grad():

        output = model(x)

    prediction = output.argmax(dim=2)[0]

    words = []

    for idx in prediction:

        word = idx_to_word.get(idx.item(), "")

        if word not in [
            "<PAD>",
            "<SOS>",
            "<EOS>"
        ]:
            words.append(word)

    return " ".join(words)


test_questions = [
    "How can I reset my password?",
    "My order is delayed",
    "How do I delete my account?",
    "Payment failed",
    "Can I change my email?"
]

print("\n==============================")
print("ВІДПОВІДІ LSTM")
print("==============================\n")

for q in test_questions:

    answer = generate_response(q)

    print("Запит:", q)
    print("Відповідь:", answer)
    print()

#Task 4

print("\nЗавантаження DistilGPT2...\n")

tokenizer = AutoTokenizer.from_pretrained(
    "distilgpt2"
)

gpt_model = AutoModelForCausalLM.from_pretrained(
    "distilgpt2"
)

print("==============================")
print("ВІДПОВІДІ")
print("==============================\n")


def ask_gpt(question):

    prompt = f"Customer: {question}\nSupport:"

    inputs = tokenizer.encode(
        prompt,
        return_tensors="pt"
    )

    outputs = gpt_model.generate(
        inputs,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return text


for q in test_questions:

    print("Запит:", q)
    print("Відповідь GPT2:")
    print(ask_gpt(q))
    print()



gpt_model.save_pretrained(
    "supportflow_gpt2"
)

tokenizer.save_pretrained(
    "supportflow_gpt2"
)

print("\nGPT2 модель збережена.")

print("\nРОБОТУ ЗАВЕРШЕНО.")